# d3blobgen low level interface

### Get blob for requests

In [ ]:

from d3blobgen.core import d3function

@d3function()
def my_add(a: int, b: int) -> int:
    return a + b

blob = my_add.get_execute_blob(1, 2)

In [6]:
print(blob)

{'script': 'a=1\nb=2\nreturn a + b\n'}


In [7]:
print(blob["script"])


a=1
b=2
return a + b



### Get plugin URL for requests

In [8]:
from d3blobgen.core import get_plugin_endpoint_url

endpoint_url = get_plugin_endpoint_url("localhost", 80)
print (endpoint_url)



http://localhost:80/api/session/python/execute


### Send execute blob to plugin endpoint

Make sure that you are running Designer for this example.

In [9]:
import requests

response = requests.post(endpoint_url, json=blob)
print(f"reponse: {response}")
print(f"json   : {response.json()}")

reponse: <Response [200]>
json   : {'status': {'code': 0, 'message': '', 'details': []}, 'd3Log': 'Python script took 6.165900000000001ms\n', 'pythonLog': '', 'returnValue': '3'}


### Parsing Response to retrieve return value

In [10]:
from d3blobgen.core import PluginResponse
from typing import Any

plugin_response: PluginResponse[Any] = PluginResponse.model_validate(response.json())
print(f"response               : {plugin_response}")
print(f"return value           : {plugin_response.returnValue}")
print(f"return value type      : {type(plugin_response.returnValue)}")
print(f"cast return value      : {plugin_response.returnCastValue(int)}")
print(f"cast return value type : {type(plugin_response.returnCastValue(int))}")

response               : status=PluginStatus(code=0, message='', details=[]) d3Log='Python script took 6.165900000000001ms\n' pythonLog='' returnValue=3
return value           : 3
return value type      : <class 'int'>
cast return value      : 3
cast return value type : <class 'int'>


### With typed execute blob

In [11]:
from requests import Response
from d3blobgen.core import PluginResponse, TypedBlob, d3function

@d3function()
def get_my_dict() -> dict[str, str]:
    return {
        "a": "hello",
        "b": "world",
    }

blob: dict[str, str] = get_my_dict.get_execute_blob()
response: Response = requests.post(endpoint_url, json=blob)
plugin_response: PluginResponse = PluginResponse.model_validate(response.json())
returnValue: Any = plugin_response.returnValue
cast_returnValue: dict[str, str] = plugin_response.returnCastValue(dict[str,str])
print(f"response               : {plugin_response}")
print(f"return value           : {plugin_response.returnValue}")
print(f"return value type      : {type(plugin_response.returnValue)}")

typed_blob: TypedBlob[dict[str, str]] = get_my_dict.get_typed_execute_blob()
response: Response = requests.post(endpoint_url, json=typed_blob.blob)
typed_plugin_response: PluginResponse[dict[str, str]] = PluginResponse[typed_blob.return_type].model_validate(response.json())
typed_returnValue: dict[str, str] = typed_plugin_response.returnValue
print(f"response               : {plugin_response}")
print(f"return value           : {plugin_response.returnValue}")
print(f"return value type      : {type(plugin_response.returnValue)}")

response               : status=PluginStatus(code=0, message='', details=[]) d3Log='Python script took 7.6553ms\n' pythonLog='' returnValue={'a': 'hello', 'b': 'world'}
return value           : {'a': 'hello', 'b': 'world'}
return value type      : <class 'dict'>
response               : status=PluginStatus(code=0, message='', details=[]) d3Log='Python script took 7.6553ms\n' pythonLog='' returnValue={'a': 'hello', 'b': 'world'}
return value           : {'a': 'hello', 'b': 'world'}
return value type      : <class 'dict'>


### With helper utilities

In [14]:
from typing import Any
from d3blobgen.core import PluginResponse, d3function
from d3blobgen.utils import (
    d3_api_plugin,
    d3_api_typed_plugin
)

@d3function()
def get_my_list() -> list[str]:
    return ["Hello", "World"]

response: PluginResponse[Any] = d3_api_plugin("localhost", 80, get_my_list.get_execute_blob())
returnValue: Any = response.returnValue
castReturnValue: list[str] = response.returnCastValue(list[str])
print(f"response                : {response}")
print(f"return value            : {returnValue}")
print(f"return value type       : {type(returnValue)}")
print(f"cast value              : {castReturnValue}")
print(f"cast value type         : {type(castReturnValue)}")

typed_response: PluginResponse[list[str]] = d3_api_typed_plugin("localhost", 80, get_my_list.get_typed_execute_blob())
typed_returnValue: list[str] = response.returnValue
print(f"typed response          : {typed_response}")
print(f"typed return value      : {typed_returnValue}")
print(f"typed return value type : {type(typed_returnValue)}")

response                : status=PluginStatus(code=0, message='', details=[]) d3Log='Python script took 6.4752ms\n' pythonLog='' returnValue=['Hello', 'World']
return value            : ['Hello', 'World']
return value type       : <class 'list'>
cast value              : ['Hello', 'World']
cast value type         : <class 'list'>
typed response          : status=PluginStatus(code=0, message='', details=[]) d3Log='Python script took 5.2620000000000005ms\n' pythonLog='' returnValue=['Hello', 'World']
typed return value      : ['Hello', 'World']
typed return value type : <class 'list'>


## Register d3function with module

### d3functions with module

In [20]:
from d3blobgen.core import TypedBlob

@d3function("hellomodule")
def my_module_add(a: int, b: int) -> int:
    return a + b

@d3function("hellomodule")
def my_add_with_note(a: int, b: int, note: str) -> str:
    return "{}, Note: {}".format(my_module_add(a, b), note)

typed_blob: TypedBlob[str] = my_add_with_note.get_typed_execute_blob(1, 2, "Hello World")
print("blob ============================")
print(typed_blob.blob)


blob ============================
{'moduleName': 'hellomodule', 'script': "return my_add_with_note(1, 2, 'Hello World')"}
reg blob ========================
{'moduleName': 'hellomodule', 'contents': "\n\ndef my_module_add(a, b):\n    return a + b\n\ndef my_add_with_note(a, b, note):\n    return '{}, Note: {}'.format(my_module_add(a, b), note)"}
reg blob contents ===============


def my_module_add(a, b):
    return a + b

def my_add_with_note(a, b, note):
    return '{}, Note: {}'.format(my_module_add(a, b), note)


### Get register blob for module

In [21]:

from d3blobgen.core import D3Function

register_blob = D3Function.get_module_register_blob("hellomodule")
print("reg blob ========================")
print(register_blob)
print("reg blob contents ===============")
print(register_blob["contents"])

reg blob ========================
{'moduleName': 'hellomodule', 'contents': "\n\ndef my_module_add(a, b):\n    return a + b\n\ndef my_add_with_note(a, b, note):\n    return '{}, Note: {}'.format(my_module_add(a, b), note)"}
reg blob contents ===============


def my_module_add(a, b):
    return a + b

def my_add_with_note(a, b, note):
    return '{}, Note: {}'.format(my_module_add(a, b), note)


In [22]:
from d3blobgen.core import D3Function


register_blob = D3Function.get_module_register_blob("module2")
print("reg blob ========================")
print(register_blob)
print("reg blob contents ===============")
print(register_blob["contents"])

reg blob ========================
{'moduleName': 'module2', 'contents': "import time\nimport datetime\n\ndef get_surface_uid_with_time(surface_name):\n    surface = resourceManager.load(Path('objects/screen2/{}.apx'.format(surface_name)), Screen2)\n    return {'name': surface.path.filename, 'uid': str(surface.uid), 'time': my_time_module2()}\n\ndef my_time_module2():\n    return str(datetime.datetime.now())\n\ndef get_typed_surface(surface_name):\n    surface = resourceManager.load(Path('objects/screen2/{}.apx'.format(surface_name)), Screen2)\n    return {'name': surface.path.filename, 'uid': surface.uid, 'time': my_time_module2()}\n\ndef will_raise_if_call_different_module_function():\n    return my_time()\n\ndef sleep_50ms():\n    time.sleep(0.05)\n    return 'after 50ms'"}
reg blob contents ===============
import time
import datetime

def get_surface_uid_with_time(surface_name):
    surface = resourceManager.load(Path('objects/screen2/{}.apx'.format(surface_name)), Screen2)
    retu

In [23]:
from examples.e4_lower_level_api.lower_level_api_blob import (
    my_time_module2,
    get_typed_surface,
    Surface
)

typed_blob1: TypedBlob[str] = my_time_module2.get_typed_execute_blob()
print("blob ============================")
print(typed_blob.blob)

typed_blob2: TypedBlob[Surface] = get_typed_surface.get_typed_execute_blob("surface 1")
print("blob ============================")
print(typed_blob2.blob)

blob ============================
{'moduleName': 'hellomodule', 'script': "return my_add_with_note(1, 2, 'Hello World')"}
blob ============================
{'moduleName': 'module2', 'script': "return get_typed_surface('surface 1')"}
